<a href="https://colab.research.google.com/github/yazandarwish264-glitch/m4u3-construction-detection/blob/main/notebooks/01_training_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Training and Evaluation

**Construction element detection for site progress verification · MAICEN0526 M4U3 · Group 6**

This notebook trains a YOLOv8 detector on our Roboflow dataset, evaluates it, and writes every artefact the assignment needs into `results/`.

---

### Before you run

1. **Set the GPU.** `Runtime → Change runtime type → T4 GPU → Save`. Cell 2 checks this and will stop you if it is not set.
2. **Have your Roboflow API key ready.** Free account → `Settings → API Keys` → copy the **Private API Key**. The notebook prompts for it and hides it as you type. Never paste it into a cell.
3. **Run it properly.** `Runtime → Restart session and run all`. Not cell by cell. The whole point of this notebook is that it survives a cold start.

### What it produces

| Output | Path |
|---|---|
| Trained weights | `results/weights/best.pt` |
| Metrics, machine-readable | `results/metrics.json` |
| Training curves, PR curve, confusion matrix | `results/curves/` |
| Ground-truth annotation examples | `results/evidence/annotations/` |
| Full environment capture | `results/pip_freeze.txt` |
| Everything, zipped for download | `m4u3_results.zip` |

**Expected runtime:** about **7 minutes** on a free-tier T4 at 30 epochs (measured 6.4 min on 2026-09-21).

---
## 1 · Configuration

**This is the only cell you need to edit.** Everything downstream reads from here.

In [ ]:
# ---- CONFIG ----------------------------------------------------------------

# Roboflow dataset. Already set to our project - no edit needed.
# Confirm on the dataset page: Versions -> Download -> "Show download code".
RF_WORKSPACE = "yazan-darwish"
RF_PROJECT   = "construction-site-km7bh-fapwu"
RF_VERSION   = 1                      # generated 2026-09-21, 80/20, 640x640, no augmentation

# Model and training
MODEL_VARIANT = "yolov8s.pt"          # "yolov8n.pt" = faster, "yolov8s.pt" = balanced
EPOCHS        = 30                    # assignment minimum is 30
IMGSZ         = 640
BATCH         = 16
SEED          = 0                     # fixed so the run is reproducible
PATIENCE      = 15                    # early-stopping patience

# Fallback mode. Set True ONLY if no GPU is available.
# It trains 5 epochs to prove the pipeline runs, then loads the real released
# weights for all metrics and inference. The notebook says loudly which mode it used.
VERIFICATION_RUN = False

# Released weights, used by VERIFICATION_RUN and by notebook 02.
WEIGHTS_URL = "https://github.com/yazandarwish264-glitch/m4u3-construction-detection/releases/download/v1.0/best.pt"

# Pin ultralytics for exact reproducibility.
# Leave as None on the FIRST run, then read the version this notebook prints,
# put it here, and record it in the README reproducibility checklist.
ULTRALYTICS_PIN = "8.4.157"           # pinned after the 2026-09-21 run

# ---- END CONFIG ------------------------------------------------------------

import os
CLASSES = ["brick", "excavator", "pvcpipe", "scaffold", "steelbar"]
if VERIFICATION_RUN:
    EPOCHS = 5
print("Config loaded.")
print(f"  mode            : {'VERIFICATION RUN (5 epochs)' if VERIFICATION_RUN else 'FULL RUN'}")
print(f"  model           : {MODEL_VARIANT}")
print(f"  epochs          : {EPOCHS}")
print(f"  imgsz / batch   : {IMGSZ} / {BATCH}")
print(f"  seed            : {SEED}")

---
## 2 · Environment check

Confirms the GPU is live and records what hardware this run used. **If this prints `NO GPU`, stop and change the runtime type** — do not continue on CPU.

In [ ]:
import subprocess, sys, platform, datetime, json, os

RUN_INFO = {}
RUN_INFO["run_started_utc"] = datetime.datetime.utcnow().isoformat(timespec="seconds") + "Z"
RUN_INFO["python"] = sys.version.split()[0]
RUN_INFO["platform"] = platform.platform()

gpu_name = None
try:
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=30
    )
    if out.returncode == 0 and out.stdout.strip():
        gpu_name = out.stdout.strip().splitlines()[0]
except Exception:
    pass

RUN_INFO["accelerator"] = gpu_name or "CPU (no GPU detected)"

print("=" * 66)
print(f"Python      : {RUN_INFO['python']}")
print(f"Platform    : {RUN_INFO['platform']}")
print(f"Accelerator : {RUN_INFO['accelerator']}")
print(f"Started     : {RUN_INFO['run_started_utc']}")
print("=" * 66)

if gpu_name is None:
    print()
    print("  *** NO GPU DETECTED ***")
    print()
    print("  Runtime -> Change runtime type -> T4 GPU -> Save, then re-run all.")
    print()
    print("  If no GPU is available to you at all, set VERIFICATION_RUN = True")
    print("  in cell 1 and re-run. Training 30 epochs on CPU will not finish.")
else:
    print("GPU ready.")

---
## 3 · Install

Two packages. `ultralytics` is YOLOv8; `roboflow` fetches the dataset.

The exact versions installed are printed below and written to `results/pip_freeze.txt`. `ULTRALYTICS_PIN` is already set to the version that produced the reported results — that is what makes the run reproducible for a third party rather than merely repeatable for you.

In [ ]:
pkg = f"ultralytics=={ULTRALYTICS_PIN}" if ULTRALYTICS_PIN else "ultralytics"
!pip install -q {pkg} roboflow

import ultralytics, roboflow, torch
ultralytics.checks()

RUN_INFO["ultralytics"] = ultralytics.__version__
RUN_INFO["roboflow"] = roboflow.__version__
RUN_INFO["torch"] = torch.__version__

print()
print(f"ultralytics : {ultralytics.__version__}")
print(f"roboflow    : {roboflow.__version__}")
print(f"torch       : {torch.__version__}")
print()
print(">>> Put this in ULTRALYTICS_PIN in cell 1 and in the README:")
print(f"    ULTRALYTICS_PIN = \"{ultralytics.__version__}\"")

---
## 4 · Output folders

Created up front so no later cell fails on a missing directory.

In [ ]:
from pathlib import Path

ROOT     = Path("/content/m4u3")
RESULTS  = ROOT / "results"
CURVES   = RESULTS / "curves"
EVIDENCE = RESULTS / "evidence"
WEIGHTS  = RESULTS / "weights"

for d in [ROOT, RESULTS, CURVES, EVIDENCE / "annotations",
          EVIDENCE / "validation", EVIDENCE / "new_images", WEIGHTS]:
    d.mkdir(parents=True, exist_ok=True)

os.chdir(ROOT)
print(f"Working directory: {ROOT}")
for d in sorted(ROOT.rglob("*")):
    if d.is_dir():
        print(f"  {d.relative_to(ROOT)}/")

---
## 5 · Download the dataset - no credentials needed

The frozen dataset is published as a **GitHub Release asset**, so this notebook runs
top to bottom in a fresh Colab session with **no Roboflow account, no API key and no
Colab Secrets**. The SHA-256 below is checked on download: if the file that arrives is
not the file we trained on, the cell stops rather than training on something else.

Roboflow remains our annotation and versioning tool - see §5.2 for the optional
key-based path - but nothing in this notebook depends on it.


In [ ]:
# ---- PRIMARY DATA PATH: keyless, checksum-verified ------------------------
import hashlib, zipfile, urllib.request, yaml

DATA_URL = ("https://github.com/yazandarwish264-glitch/m4u3-construction-detection"
            "/releases/download/v1.0/construction-site-v1-yolo11.zip")
SHA256   = "26b21198babe59ebb03c5fc43fee4d956690dfebb1802bb6325f219b234dcb4e"

DATASET_DIR = ROOT / "dataset"
ZIP_PATH    = ROOT / "dataset.zip"


def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def find_data_yaml(root: Path) -> Path:
    hits = list(root.rglob("data.yaml"))
    if not hits:
        raise FileNotFoundError(f"No data.yaml under {root}")
    return hits[0]


already_ready = (DATASET_DIR / "data.yaml").exists() or any(DATASET_DIR.glob("*/data.yaml"))
if not already_ready:
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    if ZIP_PATH.exists():
        ZIP_PATH.unlink()
    print(f"Downloading {DATA_URL.rsplit('/', 1)[-1]} ...")
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)

    digest = sha256_file(ZIP_PATH)
    if digest != SHA256:
        ZIP_PATH.unlink(missing_ok=True)
        raise AssertionError(
            f"Checksum mismatch.\n  expected {SHA256}\n  got      {digest}\n"
            "The partial file has been deleted. Re-run this cell."
        )
    print(f"SHA-256 verified: {digest}")

    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATASET_DIR)
    print(f"Extracted to {DATASET_DIR}")
else:
    print("Dataset already present - skipping download.")

# Roboflow writes relative paths such as ../train/images, and Ultralytics may resolve a
# relative `path` against its own datasets folder. Pin it to the real extract root.
cfg_path    = find_data_yaml(DATASET_DIR)
DATASET_DIR = cfg_path.parent
cfg = yaml.safe_load(cfg_path.read_text())
cfg["path"]  = str(DATASET_DIR)
cfg["train"] = "train/images"
cfg["val"]   = "valid/images"
if (DATASET_DIR / "test" / "images").is_dir():
    cfg["test"] = "test/images"
else:
    cfg.pop("test", None)
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))

DATA_YAML = cfg_path
RUN_INFO["dataset"] = {"source": "github_release", "url": DATA_URL, "sha256": SHA256}

print()
print(f"data.yaml : {DATA_YAML}")
print()
print(DATA_YAML.read_text())


### 5.2 · Optional - the Roboflow path

**You do not need this cell.** It exists because Roboflow is where the dataset is
annotated and versioned, and it pins the exact version number rather than `latest`.

It is a no-op when the keyless dataset is already in place, so *Run all* never touches
it, and it fails softly rather than crashing if no key is available.


In [ ]:
# ---- SECONDARY DATA PATH: only if the keyless one did not run -------------
if not list((ROOT / "dataset").rglob("data.yaml")):
    try:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "roboflow"], check=True)
        try:
            from google.colab import userdata
            api_key = userdata.get("ROBOFLOW_API_KEY")
        except Exception:
            from getpass import getpass
            api_key = getpass("Roboflow API key (any account - the dataset is public): ")

        from roboflow import Roboflow
        rf = Roboflow(api_key=api_key)
        rf.workspace(RF_WORKSPACE).project(RF_PROJECT).version(RF_VERSION).download(
            "yolov11", location=str(ROOT / "dataset_roboflow"), overwrite=True)
        del api_key
        print("Downloaded from Roboflow into dataset_roboflow/.")
        print("NOTE: the graded results come from the keyless release asset above.")
    except Exception as e:
        print(f"Roboflow path unavailable ({type(e).__name__}: {e}) - this is not fatal.")
else:
    print("Keyless dataset already present - skipping Roboflow download.")


### 5.1 · Verify the split and class balance

Counts the images and the label instances per class. **Read this output** — the class balance here explains most of what you will later see in the error analysis. A class with very few instances will underperform, and that is a data fact, not a model fact.

In [ ]:
import yaml, collections

cfg = yaml.safe_load(DATA_YAML.read_text())
names = cfg.get("names", CLASSES)
if isinstance(names, dict):
    names = [names[k] for k in sorted(names)]

base = DATASET_DIR
counts, inst = {}, collections.Counter()

for split in ["train", "valid", "test"]:
    img_dir = base / split / "images"
    lbl_dir = base / split / "labels"
    if not img_dir.exists():
        continue
    imgs = [p for p in img_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
    counts[split] = len(imgs)
    if split == "train":
        for lp in lbl_dir.glob("*.txt"):
            for line in lp.read_text().split("\n"):
                if line.strip():
                    inst[int(line.split()[0])] += 1

total = sum(counts.values())
print("IMAGES")
for s, n in counts.items():
    print(f"  {s:<6} {n:>5}   ({n/total*100:.1f}%)")
print(f"  {'TOTAL':<6} {total:>5}")

print()
print("TRAINING INSTANCES PER CLASS")
tot_i = sum(inst.values()) or 1
for i, nm in enumerate(names):
    n = inst.get(i, 0)
    bar = "#" * int(40 * n / max(inst.values() or [1]))
    flag = "  <-- under 5%, expect weak performance" if n / tot_i < 0.05 else ""
    print(f"  {i} {nm:<22} {n:>5}  ({n/tot_i*100:5.1f}%) {bar}{flag}")
print(f"  {'':<24} {tot_i:>5}  total")

RUN_INFO["images"] = counts
RUN_INFO["train_instances"] = {names[i]: inst.get(i, 0) for i in range(len(names))}
RUN_INFO["classes"] = names

---
## 6 · Annotation examples (evidence pack, part 1)

Renders the **ground-truth** boxes onto five training images. This is the assignment's "3–5 annotation examples" requirement, generated from the actual label files rather than screenshotted — which means it cannot drift out of sync with the data.

Look at these. If a box here looks wrong, the label is wrong, and no amount of training will fix it.

In [ ]:
import cv2, random
import matplotlib.pyplot as plt

random.seed(SEED)
PALETTE = [(239, 68, 68), (34, 197, 94), (59, 130, 246), (234, 179, 8), (168, 85, 247)]

def draw_gt(img_path, lbl_path):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    if lbl_path.exists():
        for line in lbl_path.read_text().split("\n"):
            p = line.split()
            if len(p) < 5:
                continue
            c = int(p[0]); xc, yc, bw, bh = map(float, p[1:5])
            x1, y1 = int((xc - bw/2) * w), int((yc - bh/2) * h)
            x2, y2 = int((xc + bw/2) * w), int((yc + bh/2) * h)
            col = PALETTE[c % len(PALETTE)]
            cv2.rectangle(img, (x1, y1), (x2, y2), col, 2)
            label = names[c] if c < len(names) else str(c)
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
            cv2.rectangle(img, (x1, max(0, y1 - th - 6)), (x1 + tw + 6, y1), col, -1)
            cv2.putText(img, label, (x1 + 3, max(10, y1 - 4)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
    return img

train_imgs = sorted((base / "train" / "images").glob("*"))
picks = random.sample(train_imgs, min(5, len(train_imgs)))

fig, axes = plt.subplots(1, len(picks), figsize=(5 * len(picks), 5))
axes = [axes] if len(picks) == 1 else axes
for ax, ip in zip(axes, picks):
    lp = base / "train" / "labels" / (ip.stem + ".txt")
    im = draw_gt(ip, lp)
    ax.imshow(im); ax.axis("off"); ax.set_title(ip.name, fontsize=8)
    cv2.imwrite(str(EVIDENCE / "annotations" / f"gt_{ip.name}"),
                cv2.cvtColor(im, cv2.COLOR_RGB2BGR))
plt.tight_layout()
plt.savefig(CURVES / "annotation_examples.png", dpi=110, bbox_inches="tight")
plt.show()

print(f"Saved {len(picks)} annotation examples -> results/evidence/annotations/")

---
## 7 · Train

The long cell. On a free T4 at 30 epochs expect 25–45 minutes depending on dataset size.

Ultralytics defaults are used for the optimiser and learning rate (`optimizer='auto'`, `lr0=0.01`) and are **not** overridden — documented rather than tuned, so that a third party reproducing this gets the same result.

In [ ]:
from ultralytics import YOLO
import time

if VERIFICATION_RUN:
    print("=" * 66)
    print("  VERIFICATION RUN — 5 epochs")
    print("  This proves the pipeline executes end to end.")
    print("  It is NOT the reported model. Metrics and inference below use the")
    print("  released weights from WEIGHTS_URL.")
    print("=" * 66)

t0 = time.time()

model = YOLO(MODEL_VARIANT)
results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    seed=SEED,
    patience=PATIENCE,
    project=str(ROOT / "runs"),
    name="train",
    exist_ok=True,
    plots=True,
    verbose=True,
)

train_mins = (time.time() - t0) / 60
RUN_INFO["training_minutes"] = round(train_mins, 1)
RUN_INFO["mode"] = "verification_5_epoch" if VERIFICATION_RUN else "full"
RUN_INFO["epochs_run"] = EPOCHS

print()
print(f"Training finished in {train_mins:.1f} minutes.")
print(f"Artefacts: {ROOT / 'runs' / 'train'}")

---
## 8 · Select the weights to evaluate

In a full run this is the `best.pt` just trained. In a verification run it is the released weights downloaded from `WEIGHTS_URL`, because a 5-epoch model's metrics would be meaningless and reporting them would be misleading.

In [ ]:
import shutil, urllib.request

trained = ROOT / "runs" / "train" / "weights" / "best.pt"

if VERIFICATION_RUN:
    if "PASTE" in WEIGHTS_URL:
        raise RuntimeError(
            "VERIFICATION_RUN is True but WEIGHTS_URL is not set. "
            "Create the GitHub Release, paste the asset URL into cell 1, and re-run."
        )
    eval_path = WEIGHTS / "released_best.pt"
    print(f"Downloading released weights from {WEIGHTS_URL} ...")
    urllib.request.urlretrieve(WEIGHTS_URL, eval_path)
    WEIGHTS_SOURCE = f"released weights from {WEIGHTS_URL}"
else:
    eval_path = WEIGHTS / "best.pt"
    shutil.copy(trained, eval_path)
    WEIGHTS_SOURCE = "best.pt from this run"

RUN_INFO["weights_source"] = WEIGHTS_SOURCE
size_mb = eval_path.stat().st_size / 1e6

import hashlib
sha = hashlib.sha256(eval_path.read_bytes()).hexdigest()
RUN_INFO["weights_sha256"] = sha

print(f"Evaluating : {WEIGHTS_SOURCE}")
print(f"Path       : {eval_path}")
print(f"Size       : {size_mb:.1f} MB")
print(f"SHA-256    : {sha}")
print()
print(">>> Put the size and SHA-256 in README section 7.")

---
## 9 · Evaluate

Precision, recall, mAP@50 and mAP@50–95, overall and per class. **This table goes into README section 4.**

A reminder on what these mean, because the rubric asks for interpretation, not just numbers:

- **Precision** — of what the model flagged, how much was real. Low precision = crying wolf.
- **Recall** — of what was really there, how much the model found. Low recall = silent misses.
- **mAP@50** — average precision across classes at a loose 50% overlap threshold. The headline number.
- **mAP@50–95** — the same, averaged over overlap thresholds from 50% to 95%. Much harsher, because it punishes boxes that are roughly right but not tight. Always lower. A large gap between the two means the model finds objects but localises them sloppily.

In [ ]:
eval_model = YOLO(str(eval_path))
metrics = eval_model.val(data=str(DATA_YAML), imgsz=IMGSZ, split="val", verbose=False,
                         project=str(ROOT / "runs"), name="val", exist_ok=True)

box = metrics.box
overall = {
    "precision":  float(box.mp),
    "recall":     float(box.mr),
    "mAP50":      float(box.map50),
    "mAP50_95":   float(box.map),
}

print("OVERALL (validation)")
print(f"  {'Precision':<14} {overall['precision']:.3f}")
print(f"  {'Recall':<14} {overall['recall']:.3f}")
print(f"  {'mAP@50':<14} {overall['mAP50']:.3f}")
print(f"  {'mAP@50-95':<14} {overall['mAP50_95']:.3f}")
print()

per_class = {}
print("PER CLASS")
print(f"  {'class':<24}{'P':>8}{'R':>8}{'mAP50':>9}{'mAP50-95':>11}")
print("  " + "-" * 60)
for i, ci in enumerate(box.ap_class_index):
    nm = names[int(ci)] if int(ci) < len(names) else str(ci)
    p, r, ap50, ap = box.p[i], box.r[i], box.ap50[i], box.ap[i]
    per_class[nm] = {"precision": float(p), "recall": float(r),
                     "mAP50": float(ap50), "mAP50_95": float(ap)}
    print(f"  {nm:<24}{p:>8.3f}{r:>8.3f}{ap50:>9.3f}{ap:>11.3f}")

RUN_INFO["overall"] = overall
RUN_INFO["per_class"] = per_class

### 9.1 · Success criteria check

Tests the run against the criteria stated in README section 1. Stating a target and then checking it is what separates a result from a number.

In [ ]:
checks = [
    ("S1  overall mAP@50 >= 0.50",       overall["mAP50"],                                 0.50),
    ("S2  steelbar recall >= 0.50",      per_class.get("steelbar", {}).get("recall", 0.0), 0.50),
    ("S3  brick recall >= 0.40",         per_class.get("brick", {}).get("recall", 0.0),    0.40),
]

print("SUCCESS CRITERIA")
print("-" * 56)
sc = {}
for label, got, target in checks:
    ok = got >= target
    sc[label] = {"value": round(got, 3), "target": target, "met": bool(ok)}
    print(f"  {'PASS' if ok else 'FAIL'}  {label:<34} got {got:.3f}")
print("-" * 56)

if VERIFICATION_RUN:
    print("\nNote: evaluated on released weights, not on the 5-epoch verification run.")

RUN_INFO["success_criteria"] = sc
print("\n>>> Report these honestly in the README, including the failures.")
print(">>> A documented miss scores better than an undocumented pass.")

---
## 10 · Save curves and the confusion matrix

Copies every plot Ultralytics produced into `results/curves/` and displays the two that matter most.

**Read the confusion matrix before writing the error analysis.** It tells you which class pairs are being confused, which is where your three false positives will come from.

In [ ]:
import glob
from IPython.display import Image, display

src = ROOT / "runs" / "train"
copied = []
for p in list(src.glob("*.png")) + list(src.glob("*.jpg")) + list(src.glob("*.csv")):
    shutil.copy(p, CURVES / p.name)
    copied.append(p.name)

# val() writes its own plots into a runs/val* directory
for vd in sorted(ROOT.glob("runs/val*")):
    for p in vd.glob("*.png"):
        shutil.copy(p, CURVES / f"val_{p.name}")
        copied.append(f"val_{p.name}")

print("Saved to results/curves/:")
for c in sorted(set(copied)):
    print(f"  {c}")

for key in ["results.png", "labels.jpg", "confusion_matrix_normalized.png",
            "confusion_matrix.png", "BoxPR_curve.png", "PR_curve.png"]:
    f = CURVES / key
    if f.exists():
        print(f"\n--- {key} ---")
        display(Image(str(f), width=780))

---
## 11 · Write the run record

`results/metrics.json` and `results/pip_freeze.txt`. These two files are what let somebody else check your numbers a year from now without asking you anything.

In [ ]:
import datetime

RUN_INFO["run_finished_utc"] = datetime.datetime.utcnow().isoformat(timespec="seconds") + "Z"
RUN_INFO["config"] = {
    "model_variant": MODEL_VARIANT, "epochs": EPOCHS, "imgsz": IMGSZ,
    "batch": BATCH, "seed": SEED, "patience": PATIENCE,
    "roboflow": {"workspace": RF_WORKSPACE, "project": RF_PROJECT, "version": RF_VERSION},
}

(RESULTS / "metrics.json").write_text(json.dumps(RUN_INFO, indent=2))

freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"],
                        capture_output=True, text=True).stdout
(RESULTS / "pip_freeze.txt").write_text(freeze)

print(json.dumps(RUN_INFO, indent=2))
print()
print(f"pip freeze: {len(freeze.splitlines())} packages -> results/pip_freeze.txt")

### 11.1 · Paste-ready README blocks

Copy these straight into the README. No retyping, no transcription errors.

In [ ]:
o = RUN_INFO["overall"]
print("### Overall (validation set)\n")
print("| Metric | Value |")
print("|---|---|")
print(f"| Precision | {o['precision']:.3f} |")
print(f"| Recall | {o['recall']:.3f} |")
print(f"| mAP@50 | {o['mAP50']:.3f} |")
print(f"| mAP@50-95 | {o['mAP50_95']:.3f} |")
print()
print("### Per class\n")
print("| Class | P | R | mAP@50 | mAP@50-95 |")
print("|---|---|---|---|---|")
for nm, m in RUN_INFO["per_class"].items():
    print(f"| {nm} | {m['precision']:.3f} | {m['recall']:.3f} | {m['mAP50']:.3f} | {m['mAP50_95']:.3f} |")

print()
print("---")
print()
print("## 6. Reproducibility proof\n")
print("| Field | Value |")
print("|---|---|")
print(f"| Date and time of last successful full run | {RUN_INFO['run_finished_utc']} |")
print(f"| Run mode | {RUN_INFO['mode']} |")
print(f"| Accelerator | {RUN_INFO['accelerator']} |")
print(f"| Wall-clock training time | {RUN_INFO['training_minutes']} min |")
print(f"| ultralytics version | {RUN_INFO['ultralytics']} |")
print(f"| torch version | {RUN_INFO['torch']} |")
print(f"| Weights SHA-256 | {RUN_INFO['weights_sha256'][:16]}... |")

---
## 12 · Download

Zips the weights and every result artefact. Download it, then:

1. Commit the contents of `results/` to the repository — **except `weights/`**.
2. Create a GitHub Release, attach `best.pt` to it, and paste the asset URL into README section 7 and into `WEIGHTS_URL` in both notebooks.

Weights do not go in git. A 22 MB binary in git history cannot be removed later and bloats every clone.

In [ ]:
zip_path = "/content/m4u3_results.zip"
if os.path.exists(zip_path):
    os.remove(zip_path)
shutil.make_archive("/content/m4u3_results", "zip", root_dir=str(RESULTS))

mb = os.path.getsize(zip_path) / 1e6
print(f"{zip_path}  ({mb:.1f} MB)")
print()
print("Contents:")
for p in sorted(RESULTS.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(RESULTS)}  ({p.stat().st_size/1024:.0f} KB)")

try:
    from google.colab import files
    files.download(zip_path)
    files.download(str(eval_path))
except Exception as e:
    print(f"\n(Auto-download unavailable: {e})")
    print("Download manually from the Files panel on the left.")

---
## Done

**Next:**

1. Paste the metrics tables from cell 11.1 into `README.md` sections 4 and 6.
2. Put the ultralytics version into `ULTRALYTICS_PIN` (cell 1) and into the README reproducibility checklist.
3. Create the GitHub Release with `best.pt`; update `WEIGHTS_URL` here and in notebook 02.
4. Commit `results/` to the repo (not `results/weights/`).
5. Run `02_baseline_inference.ipynb` to build the prediction evidence pack.
6. Write `docs/error_analysis.md` using `results/curves/confusion_matrix.png` and the validation predictions.

**Then re-run this notebook once more from a cold start** — `Runtime → Restart session and run all` — and record that date in README section 6. Three of the ten marks are reproducibility. Prove it rather than asserting it.